In [ ]:
from weights_cuda import WeightMatrixCUDA as WeightMatrix
from topologies import square_torus
import cupy as np
import warnings
#warnings.filterwarnings("error")


In [ ]:
def step_simulation(R, V, I, t=0, Delta=.5, Eta=-5, J=15, tau=1, dt=1e-3):
    # Heun (RK2) instead of Euler + finite guards
    fR = Delta/np.pi + 2*R*V
    fV = V**2 + Eta + J*R + I - (np.pi**2)*(R**2)  # NOTE: +J*R (match your LaTeX)
    R1 = R + dt*fR
    V1 = V + dt*fV #np.clip(dt*fV, -1e8, 1e8)
    fR1 = Delta/np.pi + 2*R1*V1
    fV1 = V1**2 + Eta + J*R1 + I - (np.pi**2)*(R1**2)
    dR = 0.5*dt*(fR + fR1)
    #np.nan_to_num(dR, copy=False, nan=0.0, posinf=1e6, neginf=-1e6)
    #dR = np.clip(dR, -R/10, 1)
    R += dR*(10-R)/10
    #R[:] = np.clip(R, 1e-5, 10)
    dV = 0.5*dt*(fV + fV1)#*(100-np.abs(V))/100
    #np.nan_to_num(dV, copy=False, nan=0.0, posinf=1e6, neginf=-1e6)
    #dV = np.clip(dV, -1, 1)
    V += dV
    #V[:] = np.clip(V, -1e2, 1e2)

    #np.nan_to_num(R, copy=False, nan=0.0, posinf=1e6, neginf=-1e6)
    #np.nan_to_num(V, copy=False, nan=0.0, posinf=1e6, neginf=-1e6)
    return t + dt, dR

def f(R, V, I, Delta=1.0, Eta=-5.0, J=15.0):
    dR = Delta/np.pi + 2*R*V
    dV = V*V + Eta + J*R + I - (np.pi**2)*(R*R)
    return dR, dV

def step_simulation(R, V, I, t, dt, Delta=1.0, Eta=-5.0, J=15.0):
    k1R, k1V = f(R, V, I, Delta, Eta, J)
    k2R, k2V = f(R + 0.5*dt*k1R, V + 0.5*dt*k1V, I, Delta, Eta, J)
    k3R, k3V = f(R + 0.5*dt*k2R, V + 0.5*dt*k2V, I, Delta, Eta, J)
    k4R, k4V = f(R + dt*k3R,     V + dt*k3V,     I, Delta, Eta, J)

    R += (dt/6)*(k1R + 2*k2R + 2*k3R + k4R)
    V += (dt/6)*(k1V + 2*k2V + 2*k3V + k4V)
    return t + dt


rk4_step = np.ElementwiseKernel(
'float32 R, float32 V, float32 I, float32 dt, float32 Delta, float32 Eta, float32 J',
    'float32 R_out, float32 V_out',
    r'''
    const float pi  = 3.14159265358979323846f;
    const float pi2 = pi * pi;

    float k1R = Delta / pi + 2.0f * R * V;
    float k1V = V * V + Eta + J * R + I - pi2 * R * R;

    float R2 = R + 0.5f * dt * k1R;
    float V2 = V + 0.5f * dt * k1V;
    float k2R = Delta / pi + 2.0f * R2 * V2;
    float k2V = V2 * V2 + Eta + J * R2 + I - pi2 * R2 * R2;

    float R3 = R + 0.5f * dt * k2R;
    float V3 = V + 0.5f * dt * k2V;
    float k3R = Delta / pi + 2.0f * R3 * V3;
    float k3V = V3 * V3 + Eta + J * R3 + I - pi2 * R3 * R3;

    float R4 = R + dt * k3R;
    float V4 = V + dt * k3V;
    float k4R = Delta / pi + 2.0f * R4 * V4;
    float k4V = V4 * V4 + Eta + J * R4 + I - pi2 * R4 * R4;

    float fac = dt / 6.0f;
    R_out = R + fac * (k1R + 2.0f * k2R + 2.0f * k3R + k4R);
    V_out = V + fac * (k1V + 2.0f * k2V + 2.0f * k3V + k4V);
    ''',
    'rk4_step_mpr_f32'
)

def step_simulation(R, V, I, t, dt, Delta=1.0, Eta=-5.0, J=15.0):
    R[:], V[:] = rk4_step(R, V, I, dt, Delta, Eta, J)
    return t + dt

import cupy as cp

syn_kernel_src = r'''
extern "C" __global__
void syn_update(
    const float* __restrict__ R,
    const float* __restrict__ U,
    const float* __restrict__ V,
    const float* __restrict__ I_decayed,
    const int*   __restrict__ children,   // length N * child_count (row-major)
    int k,
    int N,
    int child_count,
    float* __restrict__ I_out
){
    int p = blockDim.x * blockIdx.x + threadIdx.x;
    if (p >= N) return;   // one thread per parent neuron

    const float* u = U + (size_t)p * k;
    float Rp = R[p];
    float Ip = I_decayed[p];

    int base = p * child_count;

    for (int cc = 0; cc < child_count; ++cc) {
        int c = children[base + cc];

        const float* v = V + (size_t)c * k;

        float dot = 0.0f;
        for (int kk = 0; kk < k; ++kk) {
            dot += u[kk] * v[kk];
        }

        float contrib = Rp * dot - 0.5f * Ip;
        atomicAdd(&I_out[c], contrib);
    }
}
''';
syn_kernel_src = r'''
extern "C" __global__
void syn_update(
    const float* __restrict__ R,
    const float* __restrict__ U,
    const float* __restrict__ V,
    const float* __restrict__ I_decayed,
    const int*   __restrict__ children,    // (N * child_count), row-major
    int k,
    int N,
    int child_count,
    float* __restrict__ I_out
){
    int row = blockDim.x * blockIdx.x + threadIdx.x;
    if (row >= N) return;   // one thread per row

    int base = row * child_count;

    float Rp = R[row];
    float Ip = I_decayed[row];

    const float* u = U + (size_t)row * k;

    for (int cc = 0; cc < child_count; ++cc) {
        int c = children[base + cc];       // child neuron id

        const float* v = V + (size_t)c * k;

        float dot = 0.0f;
        for (int kk = 0; kk < k; ++kk) {
            dot += u[kk] * v[kk];
        }

        float contrib = Rp * dot - 0.5f * Ip;

        atomicAdd(&I_out[c], contrib);
    }
}
''';

syn_update = cp.RawKernel(syn_kernel_src, 'syn_update')


diff_src = r'''
extern "C" __global__
void diffuse_I(
    const float* __restrict__ I_in,
    float* __restrict__ I_out,
    int Z,
    float Ddt  // D * dt
){
    int idx = blockDim.x * blockIdx.x + threadIdx.x;
    int N = Z * Z;
    if (idx >= N) return;

    int i = idx / Z;
    int j = idx % Z;

    int ip = (i + 1) % Z;
    int im = (i + Z - 1) % Z;
    int jp = (j + 1) % Z;
    int jm = (j + Z - 1) % Z;

    float center = I_in[i * Z + j];
    float up     = I_in[im * Z + j];
    float down   = I_in[ip * Z + j];
    float left   = I_in[i * Z + jm];
    float right  = I_in[i * Z + jp];

    float lap = up + down + left + right - 4.0f * center;

    I_out[idx] = center + Ddt * lap;
}
''';

diffuse_I = np.RawKernel(diff_src, 'diffuse_I')

In [ ]:
import threading, queue
import ipywidgets as w

Z = 128

t = 0
weights = WeightMatrix(square_torus(Z),
                       weight_initializer=lambda **s: (0.25*np.ones(s['size'])), rank=2)
                       #weight_initializer=lambda **s: (np.random.normal(scale=0.01,loc=0.25,**s)), rank=2)
#children = np.array([*weights.network.values()])



R, V = np.zeros((2, weights.size), dtype=np.float32)
I = np.zeros(weights.size, dtype=np.float32)
weights.check = True


In [ ]:
t=0
V.fill(0)
R.fill(0)
I.fill(0)

#children = np.array([*weights.network.values()])
#parents = np.arange(weights.size)[:, None]

def update_I(I, V, weights, t, children, parents, tau=8, dt=1e-3):
    #I.fill(0)

    I *= .2
    #I[0:Z] = 4 * np.sin(np.pi * t / 20)
    #I[-Z:0] = 4 * np.sin(np.pi * t / 20)
    #I *= 0.95
    delta = (R[:,None] * weights[parents, children] - I[:,None]/2)
    #delta *= dt
    np.add.at(I, children.ravel(), delta.ravel())
    I[0:Z] += 5 * np.sin(np.pi * t / 5)
    I[-Z:] += 5 * np.sin(np.pi * (t-1)/5)
    I[Z**2//2:Z**2//2+Z] = 10 * np.sin(np.pi * (t-1/2)/5)

decay = 0.2

def update_I(I, R, t, weights,
             parents_flat, children_flat,
             Z, decay=0.33):
    # decay
    I *= decay
    I_decayed = I.copy()  # snapshot for the kernel

    # zero recurrent part (keep decayed baseline)
    # we accumulate on top of I (already decayed)
    # if you want purely recurrent overwrite, use I_rec = cp.zeros_like(I)
    # and then I[...] = I_decayed + I_rec
    # here we just add to I in-place.

    U = weights.U
    Vw = weights.V

    E = parents_flat.size
    k = U.shape[1]

    threads = 256
    blocks = (E + threads - 1) // threads

    syn_update(
        (blocks,), (threads,),
        (R, U, Vw, I_decayed,
         children_flat,
         k, E, children_flat.size//E,
         I)
    )

    # external drive (same as you had)
    #I[0:Z]        = 10 * np.sin(np.pi * t / 10)
    if t<1000:
        #I[Z**2//2+Z//2]        = 25 * np.sin(np.pi * (t - 1) / 20)
        #I[0:Z:2] = 4 * np.sin(np.pi * t /10)
        #I[-Z::2] = 4 * np.sin(np.pi * (t-4/3)/10)
        I[Z**2//2:Z**2//2+Z:2] += 4 * np.sin(np.pi * (t-2/3)/10)


import plotly.graph_objects as go, time

# initial setup
fig = go.FigureWidget()
fig.update_layout(width=500, height=500, margin=dict(l=0, r=0, b=0, t=0))

heat = fig.add_heatmap(z=np.zeros((Z,Z)).get(), colorscale="Viridis",zmin=0,zmax=8)

# make the figure a square
display(fig)
minvs = []
maxvs = []

# frame interval slider (interactive)
#frame_interval = w.IntSlider(value=100, min=1, max=1000, step=1, description="Frame N", continuous_update=True)

fig = go.FigureWidget()
fig.update_layout(width=500, height=500, margin=dict(l=0, r=0, b=0, t=0))

heat = fig.add_heatmap(
    z=np.zeros((Z, Z)).get(),
    colorscale="Viridis",
    zmin=0,
    zmax=1,
)

heat = fig.data[0]

display(fig)

frame_interval = w.IntSlider(
    value=1000,
    min=1,
    max=1000,
    step=1,
    description="Frame N",
    continuous_update=False,  # <- change to False to reduce widget spam
)

display(frame_interval)

#children = np.array([*weights.network.values()])
#parents = np.arange(weights.size)[:, None]

# parents: shape (N, 1), children: shape (N, deg)
children = np.array([*weights.network.values()], dtype=np.int32)
parents  = np.array([*weights.network.keys()], dtype=np.int32)[:, None]

parents_flat  = parents.ravel()
children_flat = children.ravel()
from time import time as get_time


N_tot = Z * Z
I_tmp = np.zeros_like(I)  # same dtype/shape
D = .05
dt = 2e-3  # whatever you use in step_simulation
Ddt = np.float32(D * dt)

threads = 256
blocks = (N_tot + threads - 1) // threads

R_prev = np.empty_like(R)


t1=get_time()
for tick in range(1000000):
    R_prev[:] = R  # save old R
    update_I(I, R, t, weights, parents_flat, children_flat, Z)
    t = step_simulation(R, V, I, t, dt=dt, Delta=1.5, Eta=-4.125)
    dR = (R - R_prev) / dt  # elementwise, on GPU
    #update_W(weights, R, dR, children)
    # discrete Laplacian in-place
    weights.plastic_step(children_flat, R, dR, lr=1e-5)
    
    diffuse_I((blocks,), (threads,),
              (I, I_tmp, Z, Ddt))
    I, I_tmp = I_tmp, I

    R = np.clip(R, 0.0, 100.0)      # or whatever makes sense
    V = np.clip(V, -500.0, 500.0)
    I = np.clip(I, -500.0, 500.0)

    
    N = max(1, int(frame_interval.value))

    if tick % N == 0:
        #print(t)

        mP_grid = R.reshape(Z, Z)
        #z_host = np.log10(1e-10+mP_grid).get()          # GPU -> CPU copy
        z_host = mP_grid.get()
        with fig.batch_update():
            heat.z = z_host
print(get_time()-t1)

In [ ]:
weights[12,13]

In [ ]:
import cupy as cp

# children_2d: (N, child_count) built from weights.network
children_2d = cp.array([*weights.network.values()], dtype=cp.int32)
N, child_count = children_2d.shape

parents_rows = cp.arange(N, dtype=cp.int32)[:, None]  # 0..N-1 as rows

def update_I_ref(I_ref, R_ref, t, weights, children_2d, Z, decay=0.2):
    # decay
    I_ref *= decay

    # recurrent term using your original vectorized logic
    W_block = weights[parents_rows, children_2d]             # shape (N, child_count)
    delta = (R_ref[:, None] * W_block - I_ref[:, None] / 2)  # shape (N, child_count)
    cp.add.at(I_ref, children_2d.ravel(), delta.ravel())

    # external drive (same as your kernel path)
    #I_ref[0:Z]        += 5 * cp.sin(cp.pi * t / 5)
    #I_ref[-Z:]        += 5 * cp.sin(cp.pi * (t-1) / 5)
    #I_ref[Z**2//2:Z**2//2+Z] = 10 * cp.sin(cp.pi * (t-0.5) / 5)
    I[Z**2//2+Z//2] = 10 * cp.sin(cp.pi * (t-0.5) / 5)


In [ ]:
syn_kernel_src = r'''
extern "C" __global__
void syn_update(
    const float* __restrict__ R,
    const float* __restrict__ U,
    const float* __restrict__ V,
    const float* __restrict__ I_decayed,
    const int*   __restrict__ children,    // (N * child_count), row-major
    int k,
    int N,
    int child_count,
    float* __restrict__ I_out
){
    int row = blockDim.x * blockIdx.x + threadIdx.x;
    if (row >= N) return;   // one thread per row

    int base = row * child_count;

    float Rp = R[row];
    float Ip = I_decayed[row];

    const float* u = U + (size_t)row * k;

    for (int cc = 0; cc < child_count; ++cc) {
        int c = children[base + cc];       // child neuron id

        const float* v = V + (size_t)c * k;

        float dot = 0.0f;
        for (int kk = 0; kk < k; ++kk) {
            dot += u[kk] * v[kk];
        }

        float contrib = Rp * dot - 0.5f * Ip;

        atomicAdd(&I_out[c], contrib);
    }
}
''';

syn_update = cp.RawKernel(syn_kernel_src, 'syn_update')

children_flat = children_2d.ravel()
N, child_count = children_2d.shape
k = weights.U.shape[1]

def update_I_kernel(I, R, t, weights, children_flat, N, child_count, Z, decay=0.2):
    I *= decay
    I_decayed = I.copy()

    U = weights.U
    Vw = weights.V

    threads = 256
    blocks = (N + threads - 1) // threads

    syn_update(
        (blocks,), (threads,),
        (R, U, Vw, I_decayed,
         children_flat,
         k, N, child_count,
         I)
    )

    #I[0:Z]        += 5 * cp.sin(cp.pi * t / 5)
    #I[-Z:]        += 5 * cp.sin(cp.pi * (t-1) / 5)
    #I[Z**2//2:Z**2//2+Z] = 10 * cp.sin(cp.pi * (t-0.5) / 5)
    I[Z**2//2+Z//2] = 10 * cp.sin(cp.pi * (t-0.5) / 5)


In [ ]:
#I.fill(0)
#R.fill(0)
I_ref = I.copy()
I_ker = I.copy()
R_ref = R.copy()
R_ker = R.copy()

t_test = t


update_I_ref(I_ref, R_ref, t_test, weights, children_2d, Z, decay=0.2)
update_I_kernel(I_ker, R_ker, t_test, weights, children_flat, N, child_count, Z, decay=0.2)

diff = cp.max(cp.abs(I_ref - I_ker))
print("max |I_ref - I_ker| =", float(diff))


In [ ]:
from matplotlib import pyplot as plt

In [ ]:
plt.imshow(I_ker.get().reshape(Z,Z), vmin=0, vmax=.1)

In [ ]:
import cupy as cp
print(cp.cuda.runtime.getDeviceCount())


In [ ]:
%conda install anywidget -y

In [ ]:
i=7
j=np.int64(list(weights.children[i]))
weights[[i],j]

In [ ]:
V

In [ ]:
weights[np.arange(10),np.arange(10,20)]

In [ ]:
# set the IOPub message limit much higher
import os, json, sys

# Increase limits for the running IPython kernel process (best effort)
os.environ["IPYKERNEL_CELL_NAME"] = "high_iopub_limits"
try:
    from IPython import get_ipython

    ip = get_ipython()
    if ip is not None and hasattr(ip, "kernel") and hasattr(ip.kernel, "session"):
        # These config keys are read at startup; for a running kernel we adjust traitlets directly if present
        if hasattr(ip.kernel, "iopub_thread") and hasattr(ip.kernel.iopub_thread, "rate_limit"):
            # Disable rate limiting by setting huge limits
            ip.kernel.iopub_thread.rate_limit = 1e10
            ip.kernel.iopub_thread.max_msg_rate = 1e9
            ip.kernel.iopub_thread.max_msg_size = int(1e9)
except Exception as e:
    print("Could not adjust IOPub limits at runtime:", e, file=sys.stderr)

# Also tell Jupyter Server (if it respects env for spawned kernels later in this session)
os.environ["IPKernelApp.iopub_msg_rate_limit"] = "1000000000"
os.environ["IPKernelApp.iopub_data_rate_limit"] = "1.0e11"


In [ ]:
ip.kernel.iopub_thread.rate_limit = 1e10

In [ ]:
os.environ["IPKernelApp.iopub_msg_rate_limit"] = "1000000000"

In [ ]:
import numpy as np, time
import plotly.graph_objects as go

fig = go.FigureWidget([go.Scatter(x=[], y=[], mode="lines")])
display(fig)

y = []
for t in range(1000):
    y.append(np.sin(t/10))
    with fig.batch_update():
        fig.data[0].x = np.arange(len(y))
        fig.data[0].y = y
    time.sleep(0.001)


In [ ]:
 import ipywidgets as w, plotly.io as pio
print("ipywidgets", w.__version__)
_ = w.IntSlider()  # should render a slider
